In [1]:
# Allow importation from anywhere in the repo
import sys
from pathlib import Path

repo_path = str(Path.cwd().parent)  
if repo_path not in sys.path:
    sys.path.append(repo_path)

repo_path = str(Path.cwd().parent.parent)
if repo_path not in sys.path:
    sys.path.append(repo_path)  

# Public importations
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Local importations
import preprocessing.preprocess as preprocess
import preprocessing.feature_engineering as feature_engineering
import models.models as models

In [2]:
X_train = pd.read_csv('../../data/X_train.csv',index_col='ROW_ID')
X_test = pd.read_csv('../../data/X_test.csv',index_col='ROW_ID')

y_train = pd.read_csv('../../data/y_train.csv',index_col='ROW_ID')

In [3]:
RET_2_features = [f'RET_2_{i}' for i in range(1,21)]
for i in range(1, 21):
    X_train[f'RET_2_{i}'] = X_train[f'RET_{i}']
    X_test[f'RET_2_{i}'] = X_test[f'RET_{i}']

Preprocessor = preprocess.preprocessor(X_train, X_test)
Preprocessor.add_beta_market()
Preprocessor.add_avg_corr_int()
Preprocessor.add_regime_rho_mean()
Preprocessor.add_pc1_loading()
Preprocessor.add_idio_vol()

X_train, X_test = Preprocessor.get_dataset()

### Baseline Linear Model Train

In [4]:
X_train_copy, X_test_copy = X_train.copy(), X_test.copy()
X_train, X_test = X_train_copy, X_test_copy

In [5]:
RET_columns = [f"RET_{i}" for i in range(1, 21)] + [f"RET_2_{i}" for i in range(1, 21)]
SIGNED_VOL_columns = [f'SIGNED_VOLUME_{i}' for i in range(1, 21)]
X_train, X_test = X_train.drop(RET_columns + SIGNED_VOL_columns, axis=1), X_test.drop(RET_columns + SIGNED_VOL_columns, axis=1)

In [6]:
train_dates = X_train['TS'].unique()
test_dates = X_test['TS'].unique()

dates_tr, dates_val = train_test_split(
    train_dates, 
    test_size=0.2, 
    random_state=42,
    shuffle=True
)

tr_mask = X_train['TS'].isin(dates_tr)
val_mask = X_train['TS'].isin(dates_val)

X_tr = X_train[tr_mask]
y_tr = y_train[tr_mask]

X_val = X_train[val_mask]
y_val = y_train[val_mask]
results_df = feature_engineering.select_features_univariate_ridge(X_tr[X_tr.columns[2:]], y_tr, X_val[X_tr.columns[2:]], y_val, alpha=1.0)

for k, line in results_df.iterrows():
    print(k, line['feature'], line['r2_val'])

0 AVG_DAILY_TURNOVER 0.0004635486582964532
3 BETA_5 0.00015777233753411224
15 AVG_CORR_UNIV_20 0.00010325441410763414
13 AVG_CORR_UNIV_10 -4.2997252918119955e-06
5 BETA_10 -5.04864990835685e-06
1 BETA_3 -4.403343863579323e-05
27 SIGMA_ID_20 -7.938074315405608e-05
28 ALLOCATIONS_SIGMA_ID_20 -9.563856812877525e-05
20 ALLOCATIONS_PC1LOAD_3 -0.00011519696084949516
22 ALLOCATIONS_PC1LOAD_5 -0.00012981019634517033
9 AVG_CORR_UNIV_3 -0.0001322101894016825
25 PC1LOAD_20 -0.000143700191316265
16 ALLOCATIONS_AVG_CORR_UNIV_20 -0.00014664261786445643
17 REGIME_RHO_MEAN_20 -0.00014664261786445643
18 ALLOCATIONS_REGIME_RHO_MEAN_20 -0.00014664261786445643
26 ALLOCATIONS_PC1LOAD_20 -0.00014715235621576284
4 ALLOCATIONS_BETA_5 -0.00014945539756183734
6 ALLOCATIONS_BETA_10 -0.00014945539756183734
8 ALLOCATIONS_BETA_20 -0.00014945539756183734
2 ALLOCATIONS_BETA_3 -0.00014945539756183734
24 ALLOCATIONS_PC1LOAD_10 -0.0001638717765988229
14 ALLOCATIONS_AVG_CORR_UNIV_10 -0.00017054081479517258
11 AVG_CORR_UN

In [7]:
mi_scores, mi_scores_normalized = feature_engineering.analyse_mutual_information(X_train[X_train.columns[2:]], 
                                                                                 np.array(y_train).ravel(), 
                                                                                 task='regression', 
                                                                                 n_neighbors=5)

In [8]:
mi_scores_normalized

ALLOCATIONS_SIGMA_ID_20           1.000000
ALLOCATIONS_PC1LOAD_20            0.576724
ALLOCATIONS_REGIME_RHO_MEAN_20    0.548939
REGIME_RHO_MEAN_20                0.548344
ALLOCATIONS_AVG_CORR_UNIV_20      0.546841
SIGMA_ID_20                       0.538688
ALLOCATIONS_PC1LOAD_10            0.525081
ALLOCATIONS_PC1LOAD_3             0.510719
ALLOCATIONS_AVG_CORR_UNIV_10      0.501595
ALLOCATIONS_PC1LOAD_5             0.481707
ALLOCATIONS_AVG_CORR_UNIV_5       0.397587
ALLOCATIONS_AVG_CORR_UNIV_3       0.261745
BETA_20                           0.218652
BETA_10                           0.144843
AVG_DAILY_TURNOVER                0.135501
AVG_CORR_UNIV_3                   0.075490
AVG_CORR_UNIV_10                  0.073678
BETA_5                            0.060843
PC1LOAD_20                        0.056415
AVG_CORR_UNIV_20                  0.047602
BETA_3                            0.023057
AVG_CORR_UNIV_5                   0.020930
PC1LOAD_5                         0.011565
PC1LOAD_10 

In [11]:
# Feature subset based on mi-score:
feature_subset = ['TS', 'ALLOCATION', 'BETA_5', 'BETA_20', 'ALLOCATIONS_SIGMA_ID_20', 'ALLOCATIONS_REGIME_RHO_MEAN_20', 'SIGMA_ID_20', 'REGIME_RHO_MEAN_20']


X_train_subset, X_test_subset = X_train[feature_subset], X_test[feature_subset]

X_train_subset

,TS,ALLOCATION,BETA_5,BETA_20,ALLOCATIONS_SIGMA_ID_20,ALLOCATIONS_REGIME_RHO_MEAN_20,SIGMA_ID_20,REGIME_RHO_MEAN_20
ROW_ID,,,,,,,,
0,DATE_0001,ALLOCATION_01,0.851878,-0.675607,0.002873,0.048763,0.003396,0.048763
1,DATE_0001,ALLOCATION_02,0.850256,1.982638,0.002873,0.048763,0.003529,0.048763
2,DATE_0001,ALLOCATION_03,-1.866194,-2.871131,0.002873,0.048763,0.004848,0.048763
3,DATE_0001,ALLOCATION_04,3.171874,2.891995,0.002873,0.048763,0.002035,0.048763
4,DATE_0001,ALLOCATION_05,0.092684,2.091208,0.002873,0.048763,0.002029,0.048763
...,...,...,...,...,...,...,...,...
180240,DATE_2773,ALLOCATION_61,-0.215075,0.194416,0.001585,0.088919,0.001040,0.088919
180241,DATE_2773,ALLOCATION_62,4.442118,3.424194,0.001585,0.088919,0.001857,0.088919
180242,DATE_2773,ALLOCATION_63,-0.853460,-0.522963,0.001585,0.088919,0.001291,0.088919


In [12]:
model = models.LinearTrainer(X_train_subset, y_train, _type='classifier')
model.specify_features(feature_subset)
train_results, val_results = model.train()

print("="*10, " BASELINE RIDGE ", "="*10)
print(f"Train acc: {np.mean(train_results)} +- {np.std(train_results)} ; Val acc: {np.mean(val_results)} +- {np.std(val_results)}")

Fold 0 done. Train acc: 0.506492335437331 ; Val acc: 0.5027858627858628
Fold 1 done. Train acc: 0.507862245959631 ; Val acc: 0.5086070686070686
Fold 2 done. Train acc: 0.5087974844512266 ; Val acc: 0.5102979902979903
Fold 3 done. Train acc: 0.5093670321168025 ; Val acc: 0.5089592605657587
Fold 4 done. Train acc: 0.5095742579948944 ; Val acc: 0.5076450535511906
==========  BASELINE RIDGE  ==========
Train acc: 0.5095742579948944 +- 0.0016584813358348997 ; Val acc: 0.5076450535511906 +- 0.00530994049889419


In [28]:
y_train_classifier = (y_train > 0).astype(int)
model = models.XGBoostTrainer(X_train_subset, y_train_classifier, _type='classifier')
model.specify_features(feature_subset)
train_results, val_results = model.train(n_estimators=95, max_depth=1, learning_rate=0.08)

print("="*10, " BASELINE XGBoost ", "="*10)
print(f"Train acc: {np.mean(train_results)} +- {np.std(train_results)} ; Val acc: {np.mean(val_results)} +- {np.std(val_results)}")

Fold 0 done. Train acc: 0.514087535548311 ; Val acc: 0.5103257103257103
Fold 1 done. Train acc: 0.5145279877921898 ; Val acc: 0.5114206514206514
Fold 2 done. Train acc: 0.5143811703775635 ; Val acc: 0.5095680295680296
Fold 3 done. Train acc: 0.5150142204184553 ; Val acc: 0.5089185936839367
Fold 4 done. Train acc: 0.5154550619866518 ; Val acc: 0.5092787238780019
==========  BASELINE XGBoost  ==========
Train acc: 0.5154550619866518 +- 0.0013574374750961774 ; Val acc: 0.5092787238780019 +- 0.0024757268476055003


### Baseline LGBM Model Train

In [33]:
model = models.LGBMTrainer(X_train_subset, y_train_classifier, _type='classifier')
model.specify_features(feature_subset)
train_results, val_results = model.train(n_estimators=90, max_depth=1, learning_rate=0.06)

print("="*10, " BASELINE LGBM ", "="*10)
print(f"Train acc: {np.mean(train_results)} +- {np.std(train_results)} ; Val acc: {np.mean(val_results)} +- {np.std(val_results)}")

[LightGBM] [Info] Number of positive: 72524, number of negative: 71646
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.038904 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1530
[LightGBM] [Info] Number of data points in the train set: 144170, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503045 -> initscore=0.012180
[LightGBM] [Info] Start training from score 0.012180
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

In [37]:
### Find best additionnal features
len(feature_subset)


from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
from itertools import combinations
import numpy as np
import pandas as pd
from sklearn import linear_model

# Liste des nouvelles features à tester
# new_features = ['VOL_RATIO_5_20', 'ALLOCATIONS_SORTINO_3', 'MEAN_REV_SIGNAL_10', 'MEAN_REV_SIGNAL_5', 'ALLOCATIONS_TREND_STRENGTH_10', 'ALLOCATIONS_TREND_STRENGTH_3', 'RET_2_1', 'SHARPE_15', 'VOL_RATIO_3_10', 'MAX_RET_10']

new_features = ['BETA_10', 'ALLOCATIONS_PC1LOAD_20', 'ALLOCATIONS_AVG_CORR_UNIV_20', 'ALLOCATIONS_PC1LOAD_10', "ALLOCATIONS_PC1LOAD_5", "ALLOCATIONS_AVG_CORR_UNIV_10", "PC1LOAD_10", "PC1LOAD_20"]


print(f"Nombre de nouvelles features disponibles : {len(new_features)}")
print(f"Nombre de combinaisons de 3 features : {len(list(combinations(new_features, 2)))}")

# Dictionnaire pour stocker les résultats
results = {}

# Génération de toutes les combinaisons de 3 features
for i, feature_combo in enumerate(combinations(new_features, 2)):
    
    # Ajout des 3 nouvelles features
    features_comb = feature_subset + list(feature_combo)
    print(features_comb)
    X_train_aug = X_train[features_comb].copy()
    
    # Préparation des données
    train_dates = X_train_aug['TS'].unique()
    X_features = X_train_aug[features_comb].fillna(0)
    X_features = X_features[X_features.columns[2:]].values
    y_values = y_train['target'].fillna(0).values
    ts_values = X_train_aug['TS'].values
    
    # KFold sur les dates
    kf = KFold(n_splits=5, random_state=42, shuffle=True)
    splits = list(kf.split(train_dates))
    
    # Mapping date -> indices de lignes
    date_to_idx = {date: np.where(ts_values == date)[0] for date in train_dates}
    
    # Stockage des scores pour cette combinaison
    train_accs = []
    test_accs = []
    
    for train_date_ids, test_date_ids in splits:
        # Récupération des indices de lignes à partir des dates
        train_idx = np.concatenate([date_to_idx[train_dates[d]] for d in train_date_ids])
        test_idx = np.concatenate([date_to_idx[train_dates[d]] for d in test_date_ids])
        
        # Données pour ce fold
        X_local_train = X_features[train_idx]
        y_local_train = y_values[train_idx]
        X_local_test = X_features[test_idx]
        y_local_test = y_values[test_idx]
        
        # Entraînement
        model = linear_model.Ridge(alpha=5e-2, fit_intercept=True)
        model.fit(X_local_train, y_local_train)
        
        # Prédictions
        train_pred_score = model.predict(X_local_train)
        test_pred_score = model.predict(X_local_test)
        
        # Conversion en labels binaires
        train_pred_label = (train_pred_score >= 0).astype(int)
        test_pred_label = (test_pred_score >= 0).astype(int)
        
        train_true_label = (y_local_train >= 0).astype(int)
        test_true_label = (y_local_test >= 0).astype(int)
        
        # Calcul des accuracies
        acc_train = accuracy_score(train_true_label, train_pred_label)
        acc_test = accuracy_score(test_true_label, test_pred_label)
        
        train_accs.append(acc_train)
        test_accs.append(acc_test)
    
    # Moyennes et écart-types
    mean_train_acc = np.mean(train_accs)
    std_train_acc = np.std(train_accs)
    mean_test_acc = np.mean(test_accs)
    std_test_acc = np.std(test_accs)
    
    # Stockage des résultats (clé = tuple des 3 features)
    combo_key = ' + '.join(feature_combo)
    results[combo_key] = {
        'features': feature_combo,
        'mean_train_acc': mean_train_acc,
        'std_train_acc': std_train_acc,
        'mean_test_acc': mean_test_acc,
        'std_test_acc': std_test_acc
    }
    
    # Affichage progressif (tous les 100 tests)
    if (i + 1) % 100 == 0:
        print(f"Progression : {i+1}/{len(list(combinations(new_features, 3)))} combinaisons testées")
    
    # Affichage détaillé pour les 5 premières
    print(f"\nCombinaison {i+1}: {feature_combo}")
    print(f"Train: {mean_train_acc:.4f} ± {std_train_acc:.4f} | Test: {mean_test_acc:.4f} ± {std_test_acc:.4f}")

# Conversion en DataFrame pour analyse finale
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('mean_test_acc', ascending=False)

print(f"\n{'='*80}")
print("TOP 20 MEILLEURES COMBINAISONS")
print(f"{'='*80}")
print(results_df.head(20).to_string())

print(f"\n{'='*80}")
print("TOP 5 DÉTAILLÉ")
print(f"{'='*80}")
for idx, (combo_name, row) in enumerate(results_df.head(5).iterrows(), 1):
    print(f"\n#{idx} - {combo_name}")
    print(f"   Features: {row['features']}")
    print(f"   Train accuracy: {row['mean_train_acc']:.4f} ± {row['std_train_acc']:.4f}")
    print(f"   Test accuracy:  {row['mean_test_acc']:.4f} ± {row['std_test_acc']:.4f}")

Nombre de nouvelles features disponibles : 8
Nombre de combinaisons de 3 features : 28
['TS', 'ALLOCATION', 'BETA_5', 'BETA_20', 'ALLOCATIONS_SIGMA_ID_20', 'ALLOCATIONS_REGIME_RHO_MEAN_20', 'SIGMA_ID_20', 'REGIME_RHO_MEAN_20', 'BETA_10', 'ALLOCATIONS_PC1LOAD_20']

Combinaison 1: ('BETA_10', 'ALLOCATIONS_PC1LOAD_20')
Train: 0.5097 ± 0.0010 | Test: 0.5069 ± 0.0068
['TS', 'ALLOCATION', 'BETA_5', 'BETA_20', 'ALLOCATIONS_SIGMA_ID_20', 'ALLOCATIONS_REGIME_RHO_MEAN_20', 'SIGMA_ID_20', 'REGIME_RHO_MEAN_20', 'BETA_10', 'ALLOCATIONS_AVG_CORR_UNIV_20']

Combinaison 2: ('BETA_10', 'ALLOCATIONS_AVG_CORR_UNIV_20')
Train: 0.5094 ± 0.0005 | Test: 0.5096 ± 0.0046
['TS', 'ALLOCATION', 'BETA_5', 'BETA_20', 'ALLOCATIONS_SIGMA_ID_20', 'ALLOCATIONS_REGIME_RHO_MEAN_20', 'SIGMA_ID_20', 'REGIME_RHO_MEAN_20', 'BETA_10', 'ALLOCATIONS_PC1LOAD_10']

Combinaison 3: ('BETA_10', 'ALLOCATIONS_PC1LOAD_10')
Train: 0.5102 ± 0.0008 | Test: 0.5086 ± 0.0044
['TS', 'ALLOCATION', 'BETA_5', 'BETA_20', 'ALLOCATIONS_SIGMA_ID_20'

In [ ]:
### Find best additional features with XGBoost Classifier

from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
from itertools import combinations
import numpy as np
import pandas as pd
from xgboost import XGBClassifier

n_features = 3

# Liste des nouvelles features à tester
new_features = ['BETA_10', 'ALLOCATIONS_PC1LOAD_20', 'ALLOCATIONS_AVG_CORR_UNIV_20', 'ALLOCATIONS_PC1LOAD_10', "ALLOCATIONS_PC1LOAD_5", "ALLOCATIONS_AVG_CORR_UNIV_10", "PC1LOAD_10", "PC1LOAD_20"]

print(f"Nombre de nouvelles features disponibles : {len(new_features)}")
print(f"Nombre de combinaisons de n features : {len(list(combinations(new_features, n_features)))}")

# Dictionnaire pour stocker les résultats
results = {}

# Génération de toutes les combinaisons de 2 features
for i, feature_combo in enumerate(combinations(new_features, n_features)):

    # Ajout des nouvelles features
    features_comb = feature_subset + list(feature_combo)
    print(features_comb)
    X_train_aug = X_train[features_comb].copy()

    # Préparation des données
    train_dates = X_train_aug['TS'].unique()
    X_features = X_train_aug[features_comb].fillna(0)
    X_features = X_features[X_features.columns[2:]].values  # Retirer 'ID' et 'TS'
    y_values = y_train['target'].fillna(0).values
    ts_values = X_train_aug['TS'].values

    # Conversion en labels binaires (0 ou 1)
    y_labels = (y_values >= 0).astype(int)

    # KFold sur les dates
    kf = KFold(n_splits=5, random_state=42, shuffle=True)
    splits = list(kf.split(train_dates))

    # Mapping date -> indices de lignes
    date_to_idx = {date: np.where(ts_values == date)[0] for date in train_dates}

    # Stockage des scores pour cette combinaison
    train_accs = []
    test_accs = []

    for train_date_ids, test_date_ids in splits:
        # Récupération des indices de lignes à partir des dates
        train_idx = np.concatenate([date_to_idx[train_dates[d]] for d in train_date_ids])
        test_idx = np.concatenate([date_to_idx[train_dates[d]] for d in test_date_ids])

        # Données pour ce fold
        X_local_train = X_features[train_idx]
        y_local_train = y_labels[train_idx]
        X_local_test = X_features[test_idx]
        y_local_test = y_labels[test_idx]

        # Entraînement XGBoost Classifier
        model = XGBClassifier(
            n_estimators=100,
            max_depth=1,
            learning_rate=0.06,
            random_state=42,
        )
        
        model.fit(X_local_train, y_local_train, verbose=False)

        # Prédictions
        train_pred_label = model.predict(X_local_train)
        test_pred_label = model.predict(X_local_test)

        # Calcul des accuracies
        acc_train = accuracy_score(y_local_train, train_pred_label)
        acc_test = accuracy_score(y_local_test, test_pred_label)

        train_accs.append(acc_train)
        test_accs.append(acc_test)

    # Moyennes et écart-types
    mean_train_acc = np.mean(train_accs)
    std_train_acc = np.std(train_accs)
    mean_test_acc = np.mean(test_accs)
    std_test_acc = np.std(test_accs)

    # Stockage des résultats
    combo_key = ' + '.join(feature_combo)
    results[combo_key] = {
        'features': feature_combo,
        'mean_train_acc': mean_train_acc,
        'std_train_acc': std_train_acc,
        'mean_test_acc': mean_test_acc,
        'std_test_acc': std_test_acc
    }

    # Affichage progressif
    print(f"\nCombinaison {i+1}/{len(list(combinations(new_features, n_features)))}: {feature_combo}")
    print(f"Train: {mean_train_acc:.4f} ± {std_train_acc:.4f} | Test: {mean_test_acc:.4f} ± {std_test_acc:.4f}")

# Conversion en DataFrame pour analyse finale
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('mean_test_acc', ascending=False)

print(f"\n{'='*80}")
print("TOP 20 MEILLEURES COMBINAISONS")
print(f"{'='*80}")
print(results_df.head(20).to_string())

print(f"\n{'='*80}")
print("TOP 5 DÉTAILLÉ")
print(f"{'='*80}")
for idx, (combo_name, row) in enumerate(results_df.head(5).iterrows(), 1):
    print(f"\n#{idx} - {combo_name}")
    print(f"   Features: {row['features']}")
    print(f"   Train accuracy: {row['mean_train_acc']:.4f} ± {row['std_train_acc']:.4f}")
    print(f"   Test accuracy:  {row['mean_test_acc']:.4f} ± {row['std_test_acc']:.4f}")
    print(f"   Overfitting gap: {row['mean_train_acc'] - row['mean_test_acc']:.4f}")


Nombre de nouvelles features disponibles : 8
Nombre de combinaisons de 2 features : 56
['TS', 'ALLOCATION', 'BETA_5', 'BETA_20', 'ALLOCATIONS_SIGMA_ID_20', 'ALLOCATIONS_REGIME_RHO_MEAN_20', 'SIGMA_ID_20', 'REGIME_RHO_MEAN_20', 'BETA_10', 'ALLOCATIONS_PC1LOAD_20', 'ALLOCATIONS_AVG_CORR_UNIV_20']



Combinaison 1/28: ('BETA_10', 'ALLOCATIONS_PC1LOAD_20', 'ALLOCATIONS_AVG_CORR_UNIV_20')
Train: 0.5153 ± 0.0010 | Test: 0.5084 ± 0.0019
['TS', 'ALLOCATION', 'BETA_5', 'BETA_20', 'ALLOCATIONS_SIGMA_ID_20', 'ALLOCATIONS_REGIME_RHO_MEAN_20', 'SIGMA_ID_20', 'REGIME_RHO_MEAN_20', 'BETA_10', 'ALLOCATIONS_PC1LOAD_20', 'ALLOCATIONS_PC1LOAD_10']

Combinaison 2/28: ('BETA_10', 'ALLOCATIONS_PC1LOAD_20', 'ALLOCATIONS_PC1LOAD_10')
Train: 0.5155 ± 0.0009 | Test: 0.5084 ± 0.0021
['TS', 'ALLOCATION', 'BETA_5', 'BETA_20', 'ALLOCATIONS_SIGMA_ID_20', 'ALLOCATIONS_REGIME_RHO_MEAN_20', 'SIGMA_ID_20', 'REGIME_RHO_MEAN_20', 'BETA_10', 'ALLOCATIONS_PC1LOAD_20', 'ALLOCATIONS_PC1LOAD_5']

Combinaison 3/28: ('BETA_10', 'ALLOCATIONS_PC1LOAD_20', 'ALLOCATIONS_PC1LOAD_5')
Train: 0.5159 ± 0.0014 | Test: 0.5079 ± 0.0023
['TS', 'ALLOCATION', 'BETA_5', 'BETA_20', 'ALLOCATIONS_SIGMA_ID_20', 'ALLOCATIONS_REGIME_RHO_MEAN_20', 'SIGMA_ID_20', 'REGIME_RHO_MEAN_20', 'BETA_10', 'ALLOCATIONS_PC1LOAD_20', 'ALLOCATIONS_AVG_CORR_U